# Neuroimage project

---

## Selected database

Resting-state fMRI for **34 younger** and **28 older** adults *(Wahlheim et al., OpenNeuro)*.

---

## Team

**Students:** Ainhoa Castaño · Eloi Delgado · Anton Tenev · Marcos Oriol · Silvia Estudillo

---

## 1: Explore the database — what is available?

### Dataset overview

**Source**

- Collected by **Wahlheim (2022)** at the University of North Carolina at Greensboro.
- Freely hosted on **[OpenNeuro](https://openneuro.org)**.
- Imaging: **structural (T1)** + **resting-state fMRI** from **62** healthy adults.

**Participants**

| Group | *N* | Age range | Mean age |
|:---|:---:|:---|:---|
| Younger adults | **34** | 18–32 y | **22.21** y |
| Older adults | **28** | 61–80 y | **69.82** y |

**Eligibility highlights**

- All **right-handed**
- No **neurological** history reported

The cohort can be downloaded from **AWS S3** with the shell command in the figure below.

---

![Database_download_command.png](images/Database_download_command.png)


### MRI data

Two modalities per participant:

---

#### Structural MRI (T1)

| Parameter | Value |
|:---|:---|
| Sequence | **MPRAGE** |
| Coverage | **192** sagittal slices |
| Resolution | **1 mm** isotropic |
| TR / TE | **2300 ms** / **2.26 ms** |

**Purpose:** Anatomical reference; cortical / subcortical detail; alignment to **MNI** space.

---

#### Resting-state fMRI (BOLD)

| Parameter | Value |
|:---|:---|
| Duration | Single **10 min** run |
| Volumes | **300** |
| TR / TE | **2000 ms** / **30 ms** |
| Slices | **32**, **4 mm** thickness |
| Instructions | Awake, **eyes open**, **no explicit task** |

**Preprocessing (high level)**

- CONN toolbox pipeline  
- Motion correction, **MNI** normalization, slice-time correction, denoising, smoothing  

**Retained scans after QA**

| Group | Approx. usable time |
|:---|:---|
| Young | **9.36** min |
| Older | **8.84** min |

Sufficient duration for **stable** resting-state connectivity estimates.

---

### Behavioral data (OSF)

Three **CSV** files aggregate trial logs + subject covariates used as behavioral outcomes alongside imaging.

---

#### `mst_fmri_msk.csv` — mnemonic similarity task (MST)

- **108** test trials / subject  
- Trial codes:

| Code | Label | Interpretation |
|:---:|:---|:---|
| **V** | Old / Exact | Repeated studied item |
| **B** | Similar / Lure | “Similar”, not identical |
| **N** | New / Foil | Unrelated foil |

> **LDI (lure discrimination index)**  
> Compares lure vs foil “similar” responses; **higher** values → sharper mnemonic discrimination between memories and perceptual lures.

---

#### `pdt_fmri.csv` — perceptual discrimination task (PDT)

- Two objects / trial · responses **V** (same) · **B** (similar) · **N** (different)

> **PDI** — perceptual analogue of LDI. Reported **no major age splits** → MST differences are framed as **memory-specific**, not generic perception.

---

#### `mst_icv.csv` — demographics & volumetrics

- Age (**years**)  
- **WM / GM / CSF** volumes (mm³) → summed to **ICV**

**Analysis note:** ICV enters models as a **covariate** so group effects are not confounded by global head/brain scaling.

---

### Preprocessed connectivity matrices

OSF ships summary connectivity in **`resultsROI_Condition001.mat`**:

| Field | Contents |
|:---|:---|
| **`Z`** | Fisher *z*-transformed connectivities (**291 × 294 × 70**) |
| **`names`** | ROI labels (Schaefer-200 + subcortex / cerebellum) |
| **`xyz`** | MNI coords for **294** parcels |
| **`SE`** | Standard errors (**70 × 294**) |
| **`DOF`** | Effective DOF (**≈85.4**) after cleaning |

---

## 2: Experimental design & published analyses

> **Tip:** Pair this section with the original paper bundled with the OpenNeuro record to interpret behavioural metrics and preprocessing choices.

---

### Sampling & exclusions

| Stage | Detail |
|:---|:---|
| **Site / institution** | University of North Carolina at Greensboro |
| **Baseline recruitment** | 36 young · 36 older — right-handed, neurologically healthy |
| **Final neuroimaging set** | **34 young**, **28 older** after rejecting runs with severe task-protocol deviations |

---

### Core research question

- Relate **intrinsic resting-state DMN connectivity** with **individual differences in mnemonic discrimination** (indexed by behavioural scores such as **LDI**).

---

### Acquisition protocol (three pillars)

1. **High-resolution anatomy (T1)**
2. **Resting-state fMRI · ~10 min · eyes open**
3. **Task fMRI** during two behavioural paradigms (described below)

---

### Behavioural paradigm A — mnemonic similarity task (MST)

| Phase | Protocol summary |
|:---|:---|
| **Encoding** | Single objects → indoor / outdoor judgments |
| **Retrieval** | Mix of repeats, perceptual lures, novel foils → **old · similar · new** responses |

Typical stimuli flow:

![Mnemonic similarity task schematic](images/mnemonic_task_1.png)

> *Reminder:* Labels **V · B · N** encode **old / lure / foil** judgments in CSV exports analysed later.

---

### Behavioural paradigm B — perceptual discrimination (PDT)

- Concurrent presentation of **two objects**
- Decide whether they are **identical**, **similar**, or **different**

![Perceptual discrimination task schematic](images/discrimination_task.png)

---

### Connectivity construction (study-specific ROI set)

**Cortex**

- **37 Schaefer** parcels warped to **MNI** space

**Hippocampus (Melbourne atlas)**

- Starts as **eight** segments (bilat. medial/lateral head, body, tail)  
- Averaged to **six** parcels by collapsing medial + lateral heads within each hemisphere  

**Outcome**

- One **43 × 43** Pearson correlation matrix / participant

---

### Behavioural DV — lure discrimination index (LDI)

```
LDI ≈ P("similar" | lure correct) − P("similar" | foil incorrect)
```

- Quantifies mnemonic **pattern separation** pressures without conflating simple old/new recognition alone.

---

### Connectome predictive modelling (CPM) linkage

Workflow (positive vs negative bundles):

| Step | Action |
|:---:|:---|
| **1** | Correlate **every ROI–ROI edge** with **LDI** |
| **2** | Threshold **α = .05**, split surviving edges into **positive** vs **negative** masks |
| **3** | **Sum** masked connectivities separately → regress **LDI** under **leave-one-out** cross-validation |

**Headline statistic**

- **209 / 903** (**23%**) edges significantly predicted mnemonic discrimination (**positive bundle** survives).

---

### Age moderation (young vs older)

Paper-reported contrasts include:

| Result | Interpretation sketch |
|:---|:---|
| Stronger DMN ↔ LDI CPM coupling in **young** | Behavioural discriminability tracks resting DMN scaffold more cleanly early in adulthood |
| **22 / 209** edges show **young > old** amplitude | Spatially heterogeneous age-related dampening |
| Elevated baseline DMN in younger adults **partly explains** mnemonic-predictive edges | Suggests both **mean connectivity** & **topology** contribute |

---

## 3: Inspect individual brains & reproduce course techniques

**Deliverables in this milestone**

| Target | Goal |
|:---|:---|
| **Qualitative QA** | Open representative subjects · sanity-check preprocessing · interpret anatomy |
| **Quantitative practise** | Re-apply notebooks from lab sessions · visualise voxel / ROI signals |
| **Bridge to Part 4** | Scripts & figures should be reusable inputs for supervised learning |

---

### Data ingestion & object-oriented scaffolding

Implementation lives chiefly inside **`explore_data.ipynb`**. We wrapped NIfTI assets into reusable classes so loaders stay declarative rather than procedural.

| Class | Responsibility |
|:---|:---|
| **`Recording`** | Holds a **NiBabel** volume + canonical **TR** metadata |
| **`Patient`** | Bundles paired **anatomical (T1)** & **functional (BOLD)** runs |
| **`PatientLoader`** | Walks the BIDS-ish folder tree & materialises **`Patient`** objects on demand |

**Why it mattered**

- Centralises path logic once instead of brittle copy–paste snippets.
- Mirrors how clinical / neuro pipelines package “one subject = one cohesive record”.

### DMN-motivated preprocessing stack

Early exploration focused on **resting-state** dynamics aligned with lectures on intrinsic networks—especially links to the **default mode**.

| Iteration | What we did |
|:---|:---|
| **Prototype** | Connectivity matrix extraction for **one exemplar participant** · manual notebook cells |
| **Production** | Encapsulated the recipe inside **`NeuroFeatureExtractor`** (see `process.py`) for repeatable batch runs |

Outcome: a deterministic pipeline spanning resampling → filtering → atlas-based parcellation → correlation matrices—the same stages later averaged across cohorts.

---


### `NeuroFeatureExtractor` — configuration snapshot

Instantiate with hyper-parameters, for example:

```python
extractor = NeuroFeatureExtractor(target_resolution=3, n_rois=200)
```

Baseline setup covers four pillars:

| Module | Behaviour |
|:---|:---|
| **Spatial resampling** | Warp functional volumes to **isotropic mm grid** (**3 mm** here) ⇒ inter-subject comparability |
| **Parcellation** | Loads **Harvard–Oxford** atlas split into `n_rois` masks (~**200 nodes** in production runs) |
| **Temporal band-pass** | **0.01 – 0.1 Hz** to suppress physiology & drift while preserving neural fluctuations |
| **Detrend · z-score** | Linear drift removal · **μ=0**, **σ=1** voxelwise stabilisation |

### ROI time-series construction

Calling **`extract_time_series`**:

- Applies the atlas over the participant’s functional volume  
- **Averages** voxels falling inside each ROI mask  
- Output shape ⇒ **(#timepoints × #ROIs)** — e.g. **frames × ~200 parcels**

### Functional connectivity estimation

**`extract_functional_connectivity`**:

- Computes a **symmetric Pearson** matrix (**96 × 96** in plotted examples)

| ρ value | Interpretation |
|:---:|:---|
| **≈ +1** | Strong **co-fluctuation** |
| **≈ −1** | **Anti-correlated** dynamics |
| **≈ 0** | Mostly **orthogonal** variability |

**Visualisations**

| Figure | Insight |
|:---|:---|
| **Hierarchical-clustered heat-map** | Block-diagonal motifs ≈ intrinsic networks (**DMN · sensorimotor · visual**) |
| **Glass-brain connectome** | Spatial embedding of strongest edges in **MNI** |
| **`plot_connectivity_matrix`** | Publication-style matrix render |

---

### Cohort-level connectivity & age contrasts

**Batch recipe**

1. Re-use **`NeuroFeatureExtractor`** inside a lightweight **for-loop** over every subject ID.  
2. Persist per-participant correlation tensors for downstream statistics.

**Group comparison**

| Step | Operation |
|:---:|:---|
| **A** | Average matrices separately for **young** vs **older** cohorts |
| **B** | Form **difference map** = **older − younger** |

**Reading the heat-map**

| Colour direction | Neurobiological reading |
|:---|:---|
| **Blue (negative Δ)** | Connections that **weaken** with advancing age |
| **Red (positive Δ)** | Connections that **strengthen** with age |

We observed **spatially structured** shifts rather than random noise—qualitatively **aligned** with the published ageing effects in the source article.

**Artifact for machine-learning block**

- Saved tensor → **`matrix_diff.npy`**  
- Becomes the **seed feature space** for Section **4** (supervised models).

**Notebook reference**

- Aggregated plots & tables → **`Connections_summary.ipynb`**

---

## 4: Machine-learning layer

**Scope**

| Branch | Question |
|:---|:---|
| **Feature engineering** | Fuse neuroimaging-derived connectivity summaries with **CSV** psychophysics |
| **Predictive modelling** | Quantify separability of **age** & **task performance** classes |
| **Uncertainty** | Bootstrap + simple inferential checks given **N = 62** |

---


### Behavioural & structural feature creation

**Primary notebook:** `BehavData.ipynb`

| Data source | Derived quantities |
|:---|:---|
| **MST trials** | **LDI** per difficulty **bin (1–3)** · RT summaries for **lures · foils · repeats** |
| **PDT trials** | **PDI** per bin · RT summaries for **similar · different · identical** pairs |
| **`mst_icv.csv`** merge | **Age_Years** · **ICV** · **GM / WM / CSF** volumes |

**Exploratory analytics**

- **Pair-plots** (colour = age group) → visual audit of potential **decision boundaries**  
- Follow-up EDA in **`connections_meanPCA_KNN.ipynb`** crossing **tabular** vs **fMRI** engineered traits for both **age** and **performance** classification probes

> **Take-away:** Several joint distributions show **cleaner bimodality** once neuroimaging summaries enter the feature pool—motivating the PCA + *k*NN stage next.

---

### Resting-state connectivity → age group (Random Forest)

**Notebook:** `RestingStateAdultYoungClassifier.ipynb`

| Step | What it does |
|:---|:---|
| **Inputs** | Pre-computed **`data/processed/*_features.npz`** windows — each file holds a **symmetric ROI × ROI** correlation matrix (**~3 min** resting-state segments) plus a **`young` / `adult`** label |
| **EDA** | Class balance checks and inspection of correlation structure before fitting |
| **Features** | Vectorise each matrix (e.g. upper-triangle entries) → fixed-length numeric predictor |
| **Model** | **Random Forest** with **train / test split**; evaluation via accuracy, confusion matrix, and related diagnostics |

---

### Age classifier — behaviour + connectivity stream

**Notebook:** `connections_meanPCA_KNN.ipynb`

#### 1 · Neuroimaging feature mining

| Action | Detail |
|:---|:---|
| Start from **ROI × ROI** time-series-derived connectivity |
| Rank **top-10** edges **most negative** Δ(age) & **top-10** **most positive** Δ(age) |
| Map coordinates → **Harvard–Oxford** anatomical labels |

**Notable biomarker**

- Recurrent **parahippocampal gyrus** involvement — consistent with prior **ageing + episodic memory** literature.

#### 2 · Multimodal merge & dimensionality reduction

- Concatenate **connectivity deltas** with **behavioural / volumetric** predictors from earlier section.  
- **PCA → first 6 components** capture **~90 %** cumulative variance.  
- 2-D projections still **partly overlapping** → proceed to non-linear **instance-based** classifier.

#### 3 · *k*-nearest neighbours tuning

| Setting | Observation |
|:---|:---|
| **k ∈ {1…20}** sweep | **With & without** PCA subspace |
| **Winner** | **`k = 1` + PCA** — local neighbourhoods tight; larger *k* injects label noise |
| **Hold-out accuracy** | **≈ 0.82** (stochasticity from PCA / split can shift slightly) |

Interpretation: **joint** behavioural + **DMN-related** connectivity structure carries **predictive signal** for chronological age cohort.

#### 4 · Uncertainty quantification (bootstrap)

| Procedure | Result |
|:---|:---|
| **10×** reshuffle · **70 / 30** split | Accuracies **> chance** every repeat |
| **One-sample *t*-test** vs **50 %** null | ***p* < .001** |
| **Mean ROC-AUC** | Slightly **< 0.7** — acceptable given **N = 62** |

> **Caveat:** Effect sizes wobble across resamples; expect **stabilisation** only with **larger independent cohorts**.

---
